# REML Ablation Experiment

This notebook runs the critical ablation experiment for the REML thesis.

**Three conditions** on a held-out sinusoidal regression task:
1. **REML-composed** — RL policy selects layers from a trained pool
2. **Random from trained pool** — random valid layers from the same trained pool
3. **Random from untrained pool** — random valid layers from a fresh Xavier-init pool

**Interpretation guide:**
- REML >> random-trained >> untrained: both pool and policy contribute
- REML ~= random-trained >> untrained: pool training does the work, not the policy
- All three similar: Adam optimizer equalizes everything

**Requirements:** Run on GPU runtime (Runtime > Change runtime type > T4 GPU)

## 1. Setup

In [ ]:
# Mount Google Drive so results persist across runtime restarts
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo (or pull latest if already cloned)
import os
if os.path.exists('/content/reml'):
    !cd /content/reml && git pull
else:
    !git clone https://github.com/mstachyra/reml.git /content/reml
%cd /content/reml

In [ ]:
# Install dependencies
!pip install -q torch numpy gymnasium stable-baselines3 sb3-contrib matplotlib scipy

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import sys
sys.path.insert(0, '/content/reml/src')

from reml import (
    get_default_config,
    set_seed,
    generate_tasks,
    LayerPool,
    InnerNetworkTask,
    REML,
)
from ablation_eval import run_ablation, plot_ablation_results, print_summary

print('Imports OK')

## 2. Configuration

Adjust these parameters as needed. The defaults below are scaled up for a
meaningful experiment (~30-60 min on Colab T4 GPU).

In [ ]:
import datetime

# ---- Experiment config ----
CONFIG = {
    'timesteps': 8000,      # RL timesteps for REML training per task
    'epochs': 4,            # REML training epochs over all tasks
    'n_tasks': 12,          # number of sinusoidal tasks (last 1 held out)
    'seed': 41,             # reproducibility
    'sb3_model': 'MaskablePPO',  # action-masked PPO for valid-only actions
}

N_TRIALS = 10        # repetitions per ablation condition
TRAIN_STEPS = 200    # Adam gradient steps per evaluation

# Save to Google Drive so results survive runtime restarts
run_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = f'/content/drive/MyDrive/reml_ablation_results/{run_ts}'

print('Configuration set.')
print(f'  REML training: {CONFIG["epochs"]} epoch(s), {CONFIG["timesteps"]} timesteps/task, {CONFIG["n_tasks"]} tasks')
print(f'  Ablation: {N_TRIALS} trials x {TRAIN_STEPS} Adam steps per condition')
print(f'  Save dir: {SAVE_DIR}')

## 3. Run the Full Ablation

In [ ]:
results = run_ablation(
    config_overrides=CONFIG,
    n_trials=N_TRIALS,
    train_steps=TRAIN_STEPS,
    save_dir=SAVE_DIR,
)

## 4. Results Visualization

In [ ]:
# Display the saved plot inline
from IPython.display import Image, display
import os

plot_path = os.path.join(SAVE_DIR, 'ablation_plot.png')
if os.path.exists(plot_path):
    display(Image(filename=plot_path))
else:
    print('Plot not found. Re-generating...')
    plot_ablation_results(results, TRAIN_STEPS, SAVE_DIR)
    display(Image(filename=plot_path))

In [ ]:
# Print summary table
print_summary(results)

## 5. Detailed Analysis

In [ ]:
import numpy as np

# Per-trial final losses
print('Per-trial final MSE losses:')
print(f'{"Trial":>6s}  {"REML":>12s}  {"Rand-Trained":>12s}  {"Rand-Untrained":>14s}')
for i in range(len(results['reml'])):
    print(f'{i+1:6d}  {results["reml"][i, -1]:12.6f}  {results["random_trained"][i, -1]:12.6f}  {results["random_untrained"][i, -1]:14.6f}')

print()
print('Statistics:')
for key, label in [('reml', 'REML'), ('random_trained', 'Rand-Trained'), ('random_untrained', 'Rand-Untrained')]:
    final = results[key][:, -1]
    print(f'  {label:20s}  mean={final.mean():.6f}  std={final.std():.6f}  min={final.min():.6f}  max={final.max():.6f}')

In [ ]:
# Statistical significance (paired t-test)
from scipy import stats

reml_final = results['reml'][:, -1]
rand_trained_final = results['random_trained'][:, -1]
rand_untrained_final = results['random_untrained'][:, -1]

t1, p1 = stats.ttest_rel(reml_final, rand_trained_final)
t2, p2 = stats.ttest_rel(rand_trained_final, rand_untrained_final)
t3, p3 = stats.ttest_rel(reml_final, rand_untrained_final)

print('Paired t-tests on final losses:')
print(f'  REML vs Random-Trained:     t={t1:.3f}, p={p1:.4f} {"***" if p1<0.001 else "**" if p1<0.01 else "*" if p1<0.05 else "n.s."}')
print(f'  Rand-Trained vs Untrained:  t={t2:.3f}, p={p2:.4f} {"***" if p2<0.001 else "**" if p2<0.01 else "*" if p2<0.05 else "n.s."}')
print(f'  REML vs Random-Untrained:   t={t3:.3f}, p={p3:.4f} {"***" if p3<0.001 else "**" if p3<0.01 else "*" if p3<0.05 else "n.s."}')

## 6. Download Results

In [ ]:
# Download all results as a zip (also saved to Google Drive)
import shutil
shutil.make_archive('/content/ablation_results', 'zip', SAVE_DIR)

try:
    from google.colab import files
    files.download('/content/ablation_results.zip')
except ImportError:
    print('Not running on Colab. Results saved to:', SAVE_DIR)